In [13]:
# Install XGBoost
!pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/72.0 MB 1.0 MB/s eta 0:01:09
   ---------------------------------------- 0.5/72.0 MB 1.0 MB/s eta 0:01:09
   ---------------------------------------- 0.5/72.0 MB 1.0 MB/s eta 0:01:09
    --------------------------------------- 1.0/72.0 MB 740.5 kB/s eta 0:01:36
    --------------------------------------- 1.0/72.0 MB 740.5 kB/s eta 0:01:36
    --------------------------------------- 1.0/72.0 MB 740.5 kB/s eta 0:01:36
    --------------------------------------- 1.0/72.0 MB 740.5 kB/s eta 0:01:36
    --------------------------------------- 1.3/72.0 MB 554.5 kB/s eta 0:02:08
    --------------------------------------- 1.3/72.0 MB 554.5 kB/s eta 0:02:08
    -----------

In [ ]:
# Install dependencies
!pip install git+https://github.com/openai/whisper.git --quiet
!pip install torchaudio --quiet
# Install required packages
!pip install sounddevice wavio imageio[ffmpeg] --quiet
!pip install xgboost

In [1]:
import os
print(os.getcwd())
audio_file = r"C:\Users\khali\OneDrive\Bureau\medical_rag\voice-model\Recording.wav"


c:\Users\khali\OneDrive\Bureau\medical_rag\voice-model


In [2]:
import os
print(os.path.exists(audio_file))


False


In [ ]:
import whisper
model = whisper.load_model("small")
result = model.transcribe(audio_file)
print(result["text"])


In [6]:
import sounddevice as sd
import wavio
import whisper

voice=["speech","cough","breath"]
for i in voice:
    duration = 5  
    fs = 16000    
    print("Recording "+ i +" ...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    audio_file = "test_"+i+".wav"
    wavio.write(audio_file, recording, fs, sampwidth=2)
    print(f"Saved {audio_file}")

    model = whisper.load_model("small")

    result = model.transcribe(audio_file,language="en")
    print("Transcription:", result["text"])


Recording speech ...
Saved test_speech.wav
Transcription:  be right. I am so sick.
Recording cough ...
Saved test_cough.wav
Transcription:  for campus. Good morning. Ladies and gentlemen you have your turn. Okay, lets go because I want to O. sentir good. I want to
Recording breath ...
Saved test_breath.wav
Transcription:  Thank you.


In [8]:
# @title 🩺 Step 3: Get Your Diagnosis
import joblib
import numpy as np
import librosa
from scipy.stats import skew, kurtosis
import os
# 1. SETUP (Load Model & Define Feature Extractor)
model_filename = 'covid_cough_detector.pkl'

if not os.path.exists(model_filename):
    raise ValueError("❌ Model file not found! Did you run the 'Save' step?")

loaded_model = joblib.load(model_filename)
BEST_THRESHOLD = 0.2204 # The value we found earlier

def extract_surgical_features(path):
    # Guarantees 16 features or Zeros
    try:
        y, sr = librosa.load(path, sr=22050, duration=4.0)
        y, _ = librosa.effects.trim(y)
        if len(y) < 1024: return np.zeros(16)

        # Features
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        delta = librosa.feature.delta(mfcc)
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)

        def stats(m):
            return [np.mean(m), np.var(m), skew(m, axis=None), kurtosis(m, axis=None)]

        feats = []
        for x in [mfcc, delta, cent, contrast]:
            feats.extend(stats(x))
            
        return np.array(feats)
    except:
        return np.zeros(16)

# 2. PROCESS YOUR FILES
print("⚙️ Analyzing your biomarkers...")

# Map your files (If you skipped one, it will auto-fill with zeros)
files = {
    'cough': 'test_cough.wav', 
    'breath': 'test_breath.wav', 
    'speech': 'test_speech.wav'
}

patient_vector = []
for mod in ['cough', 'breath', 'speech']:
    path = files[mod]
    if os.path.exists(path):
        print(f"   Found {mod}...")
        feats = extract_surgical_features(path)
    else:
        print(f"   ⚠️ Missing {mod} (Using zeros)")
        feats = np.zeros(16)
    patient_vector.extend(feats)

# Reshape for model (1 sample, 48 features)
X_input = np.array(patient_vector).reshape(1, -1)

# 3. PREDICT
probability = loaded_model.predict_proba(X_input)[:, 1][0]

# 4. RESULTS
print("\n" + "="*40)
print(f"🩺 AI DIAGNOSIS REPORT")
print("="*40)
print(f"Sickness Probability: {probability:.1%}")
print(f"Safety Threshold:     {BEST_THRESHOLD:.1%}")
print("-" * 20)

if probability > BEST_THRESHOLD:
    print("🔴 RESULT: SICK (Signs Detected)")
    print("   The model detected irregularity in your audio.")
    print("   (Note: This is an AI experiment, not medical advice.)")
else:
    print("🟢 RESULT: HEALTHY")
    print("   No significant biomarkers found.")
print("="*40)

⚙️ Analyzing your biomarkers...
   Found cough...
   Found breath...
   Found speech...

🩺 AI DIAGNOSIS REPORT
Sickness Probability: 1.0%
Safety Threshold:     22.0%
--------------------
🟢 RESULT: HEALTHY
   No significant biomarkers found.


In [1]:
import joblib
import numpy as np
import librosa
import os
from scipy.stats import skew, kurtosis

# --- CONFIGURATION ---
MODEL_FILE = 'respiratory_screener.pkl'  # Or 'covid_cough_detector.pkl'
THRESHOLD_FILE = 'threshold.txt'

# --- 1. LOAD MODEL ---
if not os.path.exists(MODEL_FILE):
    print(f"❌ Error: Model file '{MODEL_FILE}' not found.")
    exit()

print(f"Loading {MODEL_FILE}...")
model = joblib.load(MODEL_FILE)

# Try to load the saved threshold, otherwise default to the one we found
if os.path.exists(THRESHOLD_FILE):
    with open(THRESHOLD_FILE, 'r') as f:
        THRESHOLD = float(f.read().strip())
else:
    THRESHOLD = 0.22  # Default safety threshold
    
print(f"✅ System Ready. Sensitivity Threshold: {THRESHOLD:.1%}")

# --- 2. SURGICAL FEATURE EXTRACTOR ---
# (Must match training logic EXACTLY)
def get_features(path):
    try:
        # Load audio (Resample to 22050 Hz like training)
        y, sr = librosa.load(path, sr=22050, duration=4.0)
        y, _ = librosa.effects.trim(y)
        
        # Safety check for empty files
        if len(y) < 1024: return np.zeros(16)

        # Extract Raw Features
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        delta = librosa.feature.delta(mfcc)
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)

        # Statistical Flattening (Mean, Var, Skew, Kurt)
        def stats(matrix):
            if matrix.size == 0: return [0, 0, 0, 0]
            return [
                np.mean(matrix), 
                np.var(matrix), 
                skew(matrix, axis=None), 
                kurtosis(matrix, axis=None)
            ]

        # Concatenate 4 stats for each of the 4 feature types = 16 features
        features = []
        features.extend(stats(mfcc))
        features.extend(stats(delta))
        features.extend(stats(cent))
        features.extend(stats(contrast))
        
        return np.array(features)

    except Exception as e:
        print(f"⚠️ Error reading {path}: {e}")
        return np.zeros(16)

# --- 3. RUN DIAGNOSIS ---
print("\n🔍 Analyzing Audio Files...")

files = {
    'cough': 'test_cough.wav', 
    'breath': 'test_breath.wav', 
    'speech': 'test_speech.wav'
}

patient_vector = []
missing_count = 0

for mod, filename in files.items():
    if os.path.exists(filename):
        print(f"   Processing {mod}...")
        feats = get_features(filename)
    else:
        print(f"   ❌ Missing {filename} (Filling with zeros)")
        feats = np.zeros(16)
        missing_count += 1
    patient_vector.extend(feats)

if missing_count == 3:
    print("\n❌ No audio files found! Please record 'test_cough.wav' etc.")
    exit()

# Reshape for XGBoost (1 row, 48 columns)
X_input = np.array(patient_vector).reshape(1, -1)

# Predict
risk_score = model.predict_proba(X_input)[:, 1][0]

# --- 4. FINAL REPORT ---
print("\n" + "="*30)
print(" 🩺 RESPIRATORY SCREENING RESULT")
print("="*30)
print(f"Risk Score:      {risk_score:.1%}")
print(f"Alert Threshold: {THRESHOLD:.1%}")
print("-" * 30)

if risk_score > THRESHOLD:
    print("🔴 STATUS: SIGNS OF ILLNESS DETECTED")
    print("   The model detected irregularity in your biomarkers.")
else:
    print("🟢 STATUS: HEALTHY")
    print("   Your audio biomarkers are within normal range.")
print("="*30)

Loading respiratory_screener.pkl...
✅ System Ready. Sensitivity Threshold: 22.0%

🔍 Analyzing Audio Files...
   Processing cough...
   Processing breath...
   Processing speech...

 🩺 RESPIRATORY SCREENING RESULT
Risk Score:      4.7%
Alert Threshold: 22.0%
------------------------------
🟢 STATUS: HEALTHY
   Your audio biomarkers are within normal range.


In [ ]:
import joblib
import numpy as np
import librosa
import os
from scipy.stats import skew, kurtosis

# --- CONFIGURATION ---
MODEL_A = 'model_screener.pkl'
MODEL_B = 'model_specialist.pkl'
THRESHOLD_FILE = 'threshold_screener.txt'

# --- LOAD SYSTEM ---
if not os.path.exists(MODEL_A) or not os.path.exists(MODEL_B):
    print("❌ Error: Missing model files.")
    exit()

screener = joblib.load(MODEL_A)
specialist = joblib.load(MODEL_B)

with open(THRESHOLD_FILE, 'r') as f:
    SCREEN_THRESH = float(f.read().strip())

print(f"✅ Loaded 2-Stage System (Sensitivity: {SCREEN_THRESH:.1%})")

# --- FEATURE EXTRACTOR ---
def get_features(path):
    try:
        y, sr = librosa.load(path, sr=22050, duration=4.0)
        y, _ = librosa.effects.trim(y)
        if len(y) < 1024: return np.zeros(16)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        delta = librosa.feature.delta(mfcc)
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)

        def stats(m):
            return [np.mean(m), np.var(m), skew(m, axis=None), kurtosis(m, axis=None)]

        feats = []
        for x in [mfcc, delta, cent, contrast]:
            feats.extend(stats(x))
        return np.array(feats)
    except: return np.zeros(16)

# --- RUN DIAGNOSIS ---
print("\n🔍 Analyzing...")
files = {'cough': 'test_cough.wav', 'breath': 'test_breath.wav', 'speech': 'test_speech.wav'}
patient_vector = []

for mod, filename in files.items():
    if os.path.exists(filename):
        patient_vector.extend(get_features(filename))
    else:
        print(f"⚠️ Missing {filename}")
        patient_vector.extend(np.zeros(16))

X_input = np.array(patient_vector).reshape(1, -1)

# --- THE LOGIC TREE ---
# Stage 1: Screening
sick_prob = screener.predict_proba(X_input)[:, 1][0]

print("\n" + "="*40)
print("🩺 2-STAGE DIAGNOSIS REPORT")
print("="*40)
print(f"Sickness Risk: {sick_prob:.1%}")

if sick_prob < SCREEN_THRESH:
    print("🟢 RESULT: HEALTHY")
    print("   No significant respiratory anomalies.")
else:
    # Stage 2: Specialist
    covid_prob = specialist.predict_proba(X_input)[:, 1][0]
    print(f"COVID Probability (Conditional): {covid_prob:.1%}")
    print("-" * 20)
    
    if covid_prob > 0.50:
        print("🔴 RESULT: COVID-19 DETECTED")
        print("   Biomarkers match the COVID-19 profile.")
    else:
        print("🟠 RESULT: RESPIRATORY ISSUE (Non-COVID)")
        print("   Signs of illness detected, but does not match COVID profile.")
        print("   (Possible Flu, Asthma, or Cold).")
print("="*40)

✅ Loaded 2-Stage System (Sensitivity: 22.0%)

🔍 Analyzing...

🩺 2-STAGE DIAGNOSIS REPORT
Sickness Risk: 96.9%
COVID Probability (Conditional): 94.5%
--------------------
🔴 RESULT: COVID-19 DETECTED
   Biomarkers match the COVID-19 profile.
